# Splane Mouse Hypothalamus (1/2): Identify uniform spatial domain

## Data preprocessing

In [1]:
from SPACEL.setting import set_environ_seed
seed = 10
set_environ_seed(seed)
from SPACEL import Splane
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib
import anndata
import glob
import os
from natsort import natsorted

Setting environment seed: 10


Here, we will incorporate the cell type composition predicted by **Spoint** into the spatial anndata object for subsequent spatial domain identification in **Splane** using the `add_cell_type_composition` function. This function takes a DataFrame containing the cell type composition matrix as input for spot-based spatial transcriptomic data or a series of cell type annotations as input for single-cell resolution spatial transcriptomic data.

In [ ]:
# Path to the directory
data_dir = './embryo'

# Find all .h5ad files
h5ad_files = natsorted(glob.glob(os.path.join(data_dir, '*.h5ad')))
# h5ad_files
adata_list = []
# Read all .h5ad files into a list of AnnData objects
for f in h5ad_files:
    ad = sc.read(f)
    adata_list.append(ad)

Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].
Only considering the two last: ['.MOSTA', '.h5ad'].


In [3]:
ad.obs['annotation'].unique()

['Cavity', 'Epidermis', 'Connective tissue', 'Muscle', 'Adipose tissue', ..., 'Heart', 'Sympathetic nerve', 'Spinal cord', 'Brain', 'Choroid plexus']
Length: 25
Categories (25, object): ['Adipose tissue', 'Adrenal gland', 'Bone', 'Brain', ..., 'Smooth muscle', 'Spinal cord', 'Submandibular gland', 'Sympathetic nerve']

In [4]:
# Collect all unique cell types across all .h5ad files
all_cell_types = set()
for adata in adata_list:
    all_cell_types.update(adata.obs['annotation'].unique().tolist())

all_cell_types = sorted(list(all_cell_types)) 
# all_cell_types

In [5]:
for i, adata in enumerate(adata_list):    
    # Add cell type composition
    Splane.utils.add_cell_type_composition(adata, celltype_anno=adata.obs['annotation'])
    # Add missing cell type columns as 0s
    for ct in all_cell_types:
        if ct not in adata.obs.columns:
            adata.obs[ct] = 0.0
            
    # Reorder columns consistently
    ordered_cols = [c for c in adata.obs.columns if c not in all_cell_types] + all_cell_types
    adata.obs = adata.obs[ordered_cols]
    adata.obs['slice_id'] = i
    adata.uns['celltypes'] = np.array(all_cell_types, dtype=object)

In [6]:
len(adata.uns['celltypes'])

44

## Training Splane model

In this step, we initialize the Splane model by ``Splane.init_model(...)`` using the anndata object list as input. The ``n_clusters`` parameter determines the number of spatial domains to be identified. The ``k`` parameter controls the degree of neighbors considered in the model, with a larger ``k`` value resulting in more emphasis on global structure rather than local structure. The ``gnn_dropout`` parameter influences the level of smoothness in the model’s predictions, with a higher ``gnn_dropout`` value resulting in a smoother output that accommodates the sparsity of the spatial transcriptomics data. 

We train the model by ``splane.train(...)`` to obtain latent feature of each spots/cells. The parameter ``d_l`` affects the level of batch effect correction between slices. By default, ``d_l`` is ``0.2`` for spatial transcriptomics data with single cell resolution.

Then, we can identify the spatial domain to which each spot/cell belongs by ``splane.identify_spatial_domain(...)``. By default, the results will be saved in ``spatial_domain`` column in ``.obs``. If the key parameter is provided, the results will be saved in ``.obs[key]``.

In [ ]:
splane_model = Splane.init_model(adata_list, n_clusters=44,use_gpu=False,n_neighbors=25, gnn_dropout=0.5, seed=seed)
splane_model.train(d_l=0.2)
splane_model.identify_spatial_domain()
    
sc.concat(adata_list).write(f'./{seed}/embryo_E9.5_E11.5_S1_Splane.h5ad')    

Setting global seed: 10
Calculating cell type weights...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/utils.py:76: RuntimeWarning: invalid value encountered in double_scalars
  return len(x) / W * inum / z2ss


Generating GNN inputs...
Calculating largest eigenvalue of normalized graph Laplacian...
Calculating Chebyshev polynomials up to order 2...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:57: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:641.)
  return torch.sparse.FloatTensor(indices, values, shape)


Setting the weight of similarity loss to 2.900


The best epoch 90 total loss=-3.916 g loss=-3.530 d loss=1.930 d acc=0.348 simi loss=-0.988 db loss=0.857:  43%|████▎     | 129/300 [2:16:05<2:47:06, 58.64s/it]

In [ ]:
adata_list[0]